# 1 Map All Coordinate Sources
Scatter-map of every coordinate from all scraped sources, colour-coded by origin.

Input: data/1_derived/5_geocode_truck_stops/1_joined_all_sources.csv
Output: output/2_analysis/figures/coordinate_qc/ (HTML maps)

In [ ]:
from pathlib import Path
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import math


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'data').exists() and (candidate / 'source').exists() and (candidate / 'temp').exists():
            return candidate
    return start


PROJECT_ROOT = find_project_root(Path.cwd())
IN_FILE = PROJECT_ROOT / 'data' / '1_derived' / '5_geocode_truck_stops' / '1_joined_all_sources.csv'
FIG_DIR = PROJECT_ROOT / 'output' / '2_analysis' / 'figures' / 'coordinate_qc'
FIG_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(IN_FILE, low_memory=False)
print(f'Loaded {len(df):,} rows')

In [ ]:
# Extract all coordinate sources into a long-form DataFrame
latitudes, longitudes, sources = [], [], []

def extract_lat_lon(df, lat_col, lon_col, label):
    for _, row in df.iterrows():
        for x_lat, x_lon in zip(
            [v.strip() for v in str(row.get(lat_col, '')).split(';')],
            [v.strip() for v in str(row.get(lon_col, '')).split(';')]
        ):
            if x_lat.lower() not in ('nan', '', 'none', 'removed') and x_lon.lower() not in ('nan', '', 'none', 'removed'):
                try:
                    latitudes.append(float(x_lat))
                    longitudes.append(float(x_lon))
                    sources.append(label)
                except ValueError:
                    pass

extract_lat_lon(df, 'Webscraped_Phone_Latitude', 'Webscraped_Phone_Longitude', 'Truckstopsandservices (Phone)')
extract_lat_lon(df, 'Webscraped_PlacedMatched_Latitude', 'Webscraped_PlacedMatched_Longitude', 'Truckstopsandservices (PlaceMatched)')
extract_lat_lon(df, 'Yelp_Latitude', 'Yelp_Longitude', 'Yelp (Phone)')
extract_lat_lon(df, 'YellowPages_JSONLD_LAT_1', 'YellowPages_JSONLD_LNG_1', 'YellowPages (Phone)')

plot_df = pd.DataFrame({'Latitude': latitudes, 'Longitude': longitudes, 'Source': sources})
print(f'Total coordinate points: {len(plot_df):,}')

fig = px.scatter_mapbox(plot_df, lat='Latitude', lon='Longitude', color='Source',
                        zoom=3, height=900, width=1800)
fig.update_layout(mapbox_style='open-street-map', margin={'r':0,'t':0,'l':0,'b':0})
fig.show()
fig.write_html(str(FIG_DIR / '1_all_sources_map.html'))
print(f'Saved map to {FIG_DIR / "1_all_sources_map.html"}')